# Session 3: simulate real loans and validate the formulas
**A. Real-world problem.** Real loans are messy: you may pay extra, the bank may change the
rate, a "flat 8 percent" offer may hide a much higher true rate, and money loses value over
time. We build a month-by-month simulator, check it against Session 2's formulas, and use
it on these cases.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"


def emi(P, annual, months):
    """EMI = P r (1+r)^n / ((1+r)^n - 1), r = annual / 12 (Session 2)."""
    r = annual / 12.0
    if r == 0:
        return P / months
    g = (1.0 + r) ** months
    return P * r * g / (g - 1.0)

## Step 1: one simulator for everything
Each month: add interest at the current rate, then subtract the payment (and a lump sum
if one is due). Stop when the balance is paid off. If the balance has not reached zero
after `max_months`, report that the loan never clears.

In [ ]:
def simulate(P, rate_by_month, payment, lump=0.0, lump_month=None, max_months=1200):
    """rate_by_month(k) = annual rate in month k (k = 1, 2, ...).
    Returns (months_to_clear, total_interest, balances); months_to_clear is None if never cleared."""
    bal, paid_interest, hist = float(P), 0.0, [float(P)]
    for k in range(1, max_months + 1):
        i = bal * rate_by_month(k) / 12.0
        paid_interest += i
        bal = bal + i
        bal -= min(payment, bal)
        if lump_month is not None and k == lump_month:
            bal -= min(lump, bal)
        hist.append(bal)
        if bal <= 1e-6:
            return k, paid_interest, np.array(hist)
    return None, paid_interest, np.array(hist)

## Step 2: prepayment
Keep the EMI, but pay a lump sum after month 12 (or raise the EMI by 10 percent). The loan
ends earlier and the interest saved is the drop in total interest.

In [ ]:
def prepayment_case(P=200000.0, annual=0.12, months=60, lump=50000.0, lump_month=12, bump=0.10):
    """Tenure and total interest for: base, lump sum, higher EMI."""
    e = emi(P, annual, months)
    base = simulate(P, lambda k: annual, e)
    with_lump = simulate(P, lambda k: annual, e, lump, lump_month)
    higher = simulate(P, lambda k: annual, e * (1.0 + bump))
    return {"emi": e, "base": base[:2], "lump": with_lump[:2], "higher": higher[:2],
            "balances": (base[2], with_lump[2], higher[2])}

## Step 3: a floating rate
After `change_month` the bank moves the rate. Option 1: keep the EMI (the tenure changes).
Option 2: recompute the EMI for the months that remain (the tenure stays).

In [ ]:
def floating_case(P=100000.0, old=0.10, new=0.12, months=36, change_month=12):
    e0 = emi(P, old, months)
    keep = simulate(P, lambda k: old if k <= change_month else new, e0)
    bal = simulate(P, lambda k: old, e0, max_months=change_month)[2][change_month]
    e_new = emi(bal, new, months - change_month)
    b, tot = float(P), 0.0
    for k in range(1, months + 1):
        i = b * (old if k <= change_month else new) / 12.0
        tot += i
        b += i - (e0 if k <= change_month else e_new)
    return {"emi_old": e0, "emi_new": e_new, "balance_at_change": bal,
            "keep_emi": keep[:2], "recompute_interest": tot, "final_balance": b}

## Step 4: flat rate versus reducing balance
A flat 8 percent loan charges 8 percent of the ORIGINAL amount every year, for the whole
tenure, and splits (P + interest) into equal instalments. But you repay principal every
month, so you owe less than P most of the time. The true rate is the one that makes the
reducing-balance EMI equal to the flat instalment. We find it by root-finding.

In [ ]:
def flat_instalment(P, flat, months):
    return P * (1.0 + flat * months / 12.0) / months


def true_annual_rate_bisect(P, instalment, months, tol=1e-12):
    """Annual nominal rate (monthly rate * 12) with emi(P, rate, months) = instalment, by bisection."""
    lo, hi = 0.0, 5.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if emi(P, mid, months) < instalment:
            lo = mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return 0.5 * (lo + hi)


def true_annual_rate_newton(P, instalment, months, start=0.10, steps=60):
    """Same root by Newton's method with a numerical derivative."""
    x = start
    for _ in range(steps):
        f = emi(P, x, months) - instalment
        h = 1e-7
        d = (emi(P, x + h, months) - emi(P, x - h, months)) / (2 * h)
        x_new = x - f / d
        if abs(x_new - x) < 1e-13:
            return x_new
        x = x_new
    return x

## Step 5: inflation and the present value of the EMI stream
A payment k months away is worth less today. With yearly inflation `infl` the value today
of a payment E in month k is E / (1 + infl/12)^k.

In [ ]:
def present_value(payment, months, infl):
    k = np.arange(1, months + 1)
    return float(np.sum(payment / (1.0 + infl / 12.0) ** k))

## Step 6: when a loan is too big (a failure case)
Two warnings. (1) If the EMI is bigger than the share of income you can spare, the loan is
not affordable. (2) If the payment is smaller than the monthly interest, the balance grows
instead of shrinking (negative amortisation) and the loan never clears.

In [ ]:
def max_affordable_loan(income, share, annual, months):
    """Largest P whose EMI is at most income * share."""
    r = annual / 12.0
    g = (1.0 + r) ** months
    return income * share * (g - 1.0) / (r * g)


def is_negative_amortisation(P, annual, payment):
    return payment < P * annual / 12.0

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    c = prepayment_case()
    print(f"EMI {c['emi']:.2f}")
    for key in ("base", "lump", "higher"):
        print(f"{key:7s}: {c[key][0]} months, interest {c[key][1]:.2f}")
    f = floating_case()
    print(f"floating: old EMI {f['emi_old']:.2f}, new EMI {f['emi_new']:.2f}, balance at change {f['balance_at_change']:.2f}")
    print(f"keep EMI: {f['keep_emi'][0]} months, interest {f['keep_emi'][1]:.2f}; recompute: interest {f['recompute_interest']:.2f}, final balance {f['final_balance']:.4f}")
    P = 100000.0
    inst = flat_instalment(P, 0.08, 36)
    rb, rn = true_annual_rate_bisect(P, inst, 36), true_annual_rate_newton(P, inst, 36)
    print(f"flat 8 percent, 36 months: instalment {inst:.2f}, true annual rate bisect {rb:.6f}, newton {rn:.6f}")
    print(f"reducing 8 percent EMI {emi(P, 0.08, 36):.2f}, interest {36 * emi(P, 0.08, 36) - P:.2f}; flat interest {36 * inst - P:.2f}")
    e = emi(P, 0.12, 24)
    print(f"PV of 24 EMIs at 6 percent inflation: {present_value(e, 24, 0.06):.2f} (sum {24 * e:.2f})")
    print(f"max loan at 40 percent of 30000, 12 percent, 60 months: {max_affordable_loan(30000, 0.4, 0.12, 60):.2f}")
    neg = simulate(100000.0, lambda k: 0.12, 800.0, max_months=60)
    print(f"pay 800 on 100000 at 12 percent: balance after 60 months {neg[2][-1]:.2f}")

    fig, ax = plt.subplots(figsize=(6.5, 4))
    for b, lab in zip(c["balances"], ("EMI only", "plus lump sum after month 12", "EMI +10 percent")):
        ax.plot(np.arange(len(b)), b, label=lab)
    ax.set_xlabel("month")
    ax.set_ylabel("balance owed (units)")
    ax.set_title("Prepayment bends the balance curve down")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "prepayment.png", dpi=120)

    rates = np.linspace(0.01, 0.30, 100)
    fig2, ax2 = plt.subplots(figsize=(6.5, 4))
    ax2.plot(100 * rates, [emi(P, x, 36) for x in rates], label="reducing-balance EMI")
    ax2.axhline(inst, color="C3", ls="--", label="flat 8 percent instalment")
    ax2.axvline(100 * rb, color="gray", ls=":")
    ax2.set_xlabel("annual rate (percent)")
    ax2.set_ylabel("monthly payment (units)")
    ax2.set_title("The root: the reducing rate that matches the flat offer")
    ax2.legend()
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "flat_vs_reducing.png", dpi=120)

## Try it
Do the coding tasks, the README questions (F), then the checkpoint.

## Coding tasks
1. **Easy.** Use `prepayment_case(lump_month=24)` and print the months and interest saved; predict first whether it saves more or less than the month-12 lump sum.
2. **Stretch.** Write `interest_saved(lump, month)` that returns the interest saved, then print it for months 1, 12 and 36 with a 50000 lump sum.
3. **Explain.** Explain why the true rate of a flat loan is much higher than its quoted rate, using the words 'original amount' and 'balance'.

## Assessment
- **A. Conceptual.** Why does a lump sum paid early save more interest than the same lump sum paid late?
- **B. Numerical.** A flat 8 percent loan of 100000 units for 3 years: find the total interest and the instalment, and compare the interest with a reducing 8 percent loan. Is the flat loan really cheaper than its name suggests?
- **C. Graph and simulation interpretation.** In the balance graph, what happens at the month of the lump sum, and how does the later part of the curve differ from the plain loan?
- **D. Design and optimisation challenge.** Design: monthly income is 30000 units and you decide an EMI may use at most 40 percent. At 12 percent over 60 months, what is the largest loan you can take? Then explain what goes wrong if someone pays only 800 per month on a 100000 loan at 12 percent.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    P = 100000.0
    k, tot, hist = L.simulate(P, lambda m: 0.12, L.emi(P, 0.12, 24))
    checks.append(("simulator agrees with Session 2: clears in 24 months, interest 12976.33",
                   k == 24 and abs(tot - 12976.33) < 0.005 and abs(hist[-1]) < 1e-5))
    checks.append(("zero rate edge case: 1200 paid at 100 a month clears in 12 months with no interest",
                   L.simulate(1200.0, lambda m: 0.0, 100.0)[:2] == (12, 0.0)))
    c = L.prepayment_case()
    checks.append(("prepayment base: EMI 4448.89, 60 months, interest 66933.37",
                   abs(c["emi"] - 4448.89) < 0.005 and c["base"][0] == 60 and abs(c["base"][1] - 66933.37) < 0.005))
    checks.append(("worked example: a 50000 lump sum after month 12 ends the loan in 44 months with interest 42482.30",
                   c["lump"][0] == 44 and abs(c["lump"][1] - 42482.30) < 0.005))
    checks.append(("lump sum saves 24451.07 of interest", abs(c["base"][1] - c["lump"][1] - 24451.07) < 0.005))
    checks.append(("EMI + 10 percent ends in 53 months, interest 58407.24, saving 8526.13",
                   c["higher"][0] == 53 and abs(c["higher"][1] - 58407.24) < 0.005
                   and abs(c["base"][1] - c["higher"][1] - 8526.13) < 0.005))
    bl = c["balances"][1]
    checks.append(("the balance curve drops by exactly the lump sum at month 12 (kink)",
                   abs((bl[11] * 1.01 - c["emi"]) - 50000.0 - bl[12]) < 1e-6))
    checks.append(("a lump sum larger than the balance just clears the loan",
                   L.simulate(10000.0, lambda m: 0.12, 500.0, 1e9, 3)[0] == 3))
    f = L.floating_case()
    checks.append(("floating rate: EMI 3226.72 at 10 percent, balance 69925.75 after 12 months",
                   abs(f["emi_old"] - 3226.72) < 0.005 and abs(f["balance_at_change"] - 69925.75) < 0.005))
    checks.append(("floating rate: recomputed EMI at 12 percent for 24 months is 3291.65",
                   abs(f["emi_new"] - 3291.65) < 0.005))
    checks.append(("floating rate: recomputing the EMI clears the loan exactly (balance 0) with interest 17720.18",
                   abs(f["final_balance"]) < 1e-6 and abs(f["recompute_interest"] - 17720.18) < 0.005))
    checks.append(("floating rate: keeping the old EMI takes 37 months with interest 17930.75",
                   f["keep_emi"][0] == 37 and abs(f["keep_emi"][1] - 17930.75) < 0.005))
    inst = L.flat_instalment(P, 0.08, 36)
    checks.append(("hand calculation: flat 8 percent on 100000 for 3 years has interest 24000 and instalment 3444.44",
                   abs(inst * 36 - P - 24000.0) < 1e-6 and abs(inst - 3444.44) < 0.005))
    checks.append(("reducing 8 percent has EMI 3133.64 and interest 12810.92, about half of the flat interest",
                   abs(L.emi(P, 0.08, 36) - 3133.64) < 0.005 and abs(36 * L.emi(P, 0.08, 36) - P - 12810.92) < 0.005))
    rb = L.true_annual_rate_bisect(P, inst, 36)
    rn = L.true_annual_rate_newton(P, inst, 36)
    checks.append(("worked example: the true rate of 'flat 8 percent' is 14.548 percent a year (bisection)",
                   abs(rb - 0.145481) < 1e-6))
    checks.append(("Newton's method finds the same root as bisection", abs(rb - rn) < 1e-9))
    checks.append(("the root really solves the equation: emi at the true rate = flat instalment",
                   abs(L.emi(P, rb, 36) - inst) < 1e-6))
    checks.append(("a flat 0 percent loan has true rate 0",
                   abs(L.true_annual_rate_bisect(P, L.flat_instalment(P, 0.0, 36), 36)) < 1e-5))
    e = L.emi(P, 0.12, 24)
    pv = L.present_value(e, 24, 0.06)
    checks.append(("present value of 24 EMIs of 4707.35 at 6 percent inflation is 106211.25, below the 112976.33 nominal sum",
                   abs(pv - 106211.25) < 0.01 and abs(24 * e - 112976.33) < 0.01))
    checks.append(("zero inflation: present value equals the plain sum",
                   abs(L.present_value(e, 24, 0.0) - 24 * e) < 1e-6))
    checks.append(("maximum loan with an EMI of 12000 (40 percent of 30000), 12 percent, 60 months is 539460.46",
                   abs(L.max_affordable_loan(30000.0, 0.4, 0.12, 60) - 539460.46) < 0.01
                   and abs(L.emi(539460.46, 0.12, 60) - 12000.0) < 0.01))
    checks.append(("failure case: paying 800 when the monthly interest is 1000 is negative amortisation",
                   L.is_negative_amortisation(100000.0, 0.12, 800.0) and not L.is_negative_amortisation(100000.0, 0.12, 1000.01)))
    n = L.simulate(100000.0, lambda m: 0.12, 800.0, max_months=60)
    checks.append(("failure case: after 60 months of paying 800 the balance has grown to 116333.93 and the loan never clears",
                   n[0] is None and abs(n[2][-1] - 116333.93) < 0.01))
    # mini-project: three made-up offers, 300000 over 60 months
    Pm, n = 300000.0, 60
    ex, ey, ez = L.emi(Pm, 0.105, n), L.emi(Pm, 0.095, n), L.flat_instalment(Pm, 0.06, n)
    cost = {"X": n * ex - Pm + 0.01 * Pm, "Y": n * ey - Pm + 0.03 * Pm, "Z": n * ez - Pm}
    checks.append(("mini-project: total cost X 89890.21, Y 87033.50, Z 90000.00, so Y is cheapest",
                   abs(cost["X"] - 89890.21) < 0.005 and abs(cost["Y"] - 87033.50) < 0.005
                   and abs(cost["Z"] - 90000.0) < 1e-6 and min(cost, key=cost.get) == "Y"))
    eff = {k: (1 + L.true_annual_rate_bisect(Pm * (1 - fee), ins, n) / 12) ** 12 - 1
           for k, ins, fee in (("X", ex, 0.01), ("Y", ey, 0.03), ("Z", ez, 0.0))}
    checks.append(("mini-project: effective annual rates X 11.50, Y 11.37, Z 11.40 percent (fee included)",
                   all(abs(eff[k] - v) < 5e-5 for k, v in (("X", 0.11502), ("Y", 0.11370), ("Z", 0.11404)))))
    sav = {}
    for k, ins, r in (("X", ex, 0.105), ("Y", ey, 0.095)):
        b = L.simulate(Pm, lambda m: r, ins)
        a = L.simulate(Pm, lambda m: r, ins, 50000.0, 12)
        sav[k] = (b[1] - a[1], a[0])
    checks.append(("mini-project: a 50000 lump sum after month 12 saves X 22604.66 and Y 20121.47, both ending in 49 months",
                   abs(sav["X"][0] - 22604.66) < 0.005 and abs(sav["Y"][0] - 20121.47) < 0.005
                   and sav["X"][1] == 49 and sav["Y"][1] == 49))
    checks.append(("mini-project: Z has the lowest headline rate (6 percent) yet the highest total cost",
                   max(cost, key=cost.get) == "Z"))
    checks.append(("roadmap baselines: constant 10 percent over 36 months costs 16161.87; a month-1 lump saves 32313.54",
                   abs(L.emi(100000.0, 0.10, 36) * 36 - 100000.0 - 16161.87) < 0.005
                   and abs(L.prepayment_case(lump_month=1)["base"][1] - L.prepayment_case(lump_month=1)["lump"][1] - 32313.54) < 0.005))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
